# Query-Product Relevance Classification in Amazon Search
### NLP Case Study | MTech AI/ML | Symbiosis Institute of Technology

**Task:** Classify `(query, product_title)` pairs into:
- **E** — Exact match
- **S** — Substitute
- **C** — Complement
- **I** — Irrelevant

**Models:** Naive Bayes · Logistic Regression · SVM · BiLSTM · DistilBERT

**Dataset:** [Amazon ESCI Shopping Queries Dataset](https://github.com/amazon-science/esci-data) via HuggingFace (`tasksource/esci`)

---
## 0. Runtime Check
> **Enable GPU before running:** Runtime → Change runtime type → T4 GPU

In [1]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cpu':
    print('WARNING: No GPU detected. BiLSTM and DistilBERT will be very slow.')
else:
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


---
## 1. Setup
### 1.1 Clone Repository

In [2]:
!git clone https://github.com/tp-0604/query-product-relevance-classification.git 2>/dev/null || \
    (cd query-product-relevance-classification && git pull)

%cd query-product-relevance-classification
!ls

/content/query-product-relevance-classification
LICENSE  main.py  notebooks  outputs  README.md  requirements.txt  src


### 1.2 Install Dependencies

In [3]:
# Step 1: install numpy first and restart to fix binary incompatibility
import numpy as np
import sys

try:
    # If numpy loads cleanly, skip reinstall
    from numpy.random import mtrand
    print(f'numpy {np.__version__} OK — skipping reinstall')
    _numpy_ok = True
except (ValueError, ImportError):
    _numpy_ok = False

if not _numpy_ok:
    print('numpy binary conflict detected — reinstalling...')
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy>=1.26,<2.0', '--force-reinstall'], check=True)
    print('numpy reinstalled. PLEASE RESTART RUNTIME NOW:')
    print('  Runtime → Restart runtime (or Ctrl+M .)  then re-run from this cell')
    raise SystemExit('Restart required')


numpy 2.0.2 OK — skipping reinstall


In [4]:
# Step 2: install remaining dependencies (run after numpy is confirmed OK)
!pip install -q \
    'pandas>=2.0' \
    'scikit-learn>=1.4' \
    'matplotlib>=3.7' \
    'seaborn>=0.13' \
    'torch>=2.0' \
    'transformers>=4.38' \
    'datasets>=2.17' \
    'pyarrow>=14.0' \
    'nltk>=3.8' \
    'tqdm>=4.66' \
    'joblib>=1.3'
print('All dependencies installed')


All dependencies installed


### 1.3 Imports & Setup

In [5]:
import os, sys, warnings, gc
warnings.filterwarnings('ignore')

repo_path = '/content/query-product-relevance-classification'
os.chdir(repo_path)
sys.path.insert(0, repo_path)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

os.makedirs('outputs/figures', exist_ok=True)
os.makedirs('outputs/reports', exist_ok=True)
os.makedirs('outputs/models', exist_ok=True)
os.makedirs('data', exist_ok=True)
print(f'Working dir: {os.getcwd()}')
print(f'numpy {np.__version__} | pandas {pd.__version__}')


Working dir: /content/query-product-relevance-classification
numpy 2.0.2 | pandas 2.2.2


---
## 2. Load Dataset
> Streams directly from HuggingFace (`tasksource/esci`). No manual file upload needed.
> Takes ~2 minutes to collect 20,000 rows.


In [8]:
from datasets import load_dataset

LABEL_MAP = {'Exact': 'E', 'Substitute': 'S', 'Complement': 'C', 'Irrelevant': 'I'}
SAMPLE_SIZE = 20000
PER_CLASS   = SAMPLE_SIZE // 4

print(f'Streaming {SAMPLE_SIZE} rows ({PER_CLASS} per class)...')
buckets = {'E': [], 'S': [], 'C': [], 'I': []}
done = False

for split_name in ['train', 'test']:
    if done:
        break
    ds_stream = load_dataset('tasksource/esci', split=split_name, streaming=True)
    for row in ds_stream:
        if row.get('product_locale', 'us') != 'us':
            continue
        label = LABEL_MAP.get(row.get('esci_label', ''))
        if label not in buckets:
            continue
        if len(buckets[label]) < PER_CLASS:
            buckets[label].append({
                'query':         str(row.get('query', '') or ''),
                'product_title': str(row.get('product_title', '') or ''),
                'esci_label':    label,
                'split':         split_name,
            })
        if all(len(v) >= PER_CLASS for v in buckets.values()):
            done = True
            break
    del ds_stream
    gc.collect()

rows = [r for bucket in buckets.values() for r in bucket]
df_raw = pd.DataFrame(rows)
del buckets, rows
gc.collect()

print(f'Loaded {len(df_raw)} rows')
print(df_raw['esci_label'].value_counts())
df_raw[['query', 'product_title', 'esci_label']].head(3)


Streaming 20000 rows (5000 per class)...
Loaded 20000 rows
esci_label
E    5000
S    5000
C    5000
I    5000
Name: count, dtype: int64


,query,product_title,esci_label
0,bathroom fan without light,Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceil...,E
1,revent 80 cfm,Homewerks 7141-80 Bathroom Fan Integrated LED ...,E
2,revent 80 cfm,Homewerks 7140-80 Bathroom Fan Ceiling Mount E...,E


---
## 3. Exploratory Data Analysis

In [9]:
from collections import Counter
from nltk.corpus import stopwords

label_map = {'E': 'Exact', 'S': 'Substitute', 'C': 'Complement', 'I': 'Irrelevant'}
df_raw['label_name'] = df_raw['esci_label'].map(label_map)

print('=== Class Distribution ===')
print(df_raw['esci_label'].value_counts())
print(f'\nTotal samples: {len(df_raw)}')

=== Class Distribution ===
esci_label
E    5000
S    5000
C    5000
I    5000
Name: count, dtype: int64

Total samples: 20000


In [10]:
# ── Plot 1: Class distribution ─────────────────────────────────────────────────
counts = df_raw['label_name'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
axes[0].bar(counts.index, counts.values, color=colors)
axes[0].set_title('Class Distribution (Count)', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + counts.max()*0.01, f'{v:,}', ha='center', fontsize=9)

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Class Distribution (%)', fontweight='bold')

plt.suptitle('ESCI Dataset — Class Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/figures/eda_class_distribution.png')

Saved: outputs/figures/eda_class_distribution.png


In [11]:
# ── Plot 2: Text length distributions ─────────────────────────────────────────
df_raw['query_len'] = df_raw['query'].str.split().str.len()
df_raw['title_len'] = df_raw['product_title'].fillna('').str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_raw['query_len'].clip(0, 20), bins=20, color='#4C72B0', edgecolor='white')
axes[0].axvline(df_raw['query_len'].mean(), color='red', linestyle='--',
                label=f"Mean: {df_raw['query_len'].mean():.1f}")
axes[0].set_title('Query Length (words)', fontweight='bold')
axes[0].set_xlabel('Word Count')
axes[0].legend()

axes[1].hist(df_raw['title_len'].clip(0, 60), bins=30, color='#DD8452', edgecolor='white')
axes[1].axvline(df_raw['title_len'].mean(), color='red', linestyle='--',
                label=f"Mean: {df_raw['title_len'].mean():.1f}")
axes[1].set_title('Product Title Length (words)', fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].legend()

plt.suptitle('Text Length Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/eda_text_lengths.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/figures/eda_text_lengths.png')

Saved: outputs/figures/eda_text_lengths.png


In [12]:
# ── Plot 3: Query length per class ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
df_raw.boxplot(column='query_len', by='label_name', ax=ax,
               boxprops=dict(color='#4C72B0'),
               medianprops=dict(color='red', linewidth=2),
               whiskerprops=dict(color='#4C72B0'),
               capprops=dict(color='#4C72B0'))
ax.set_title('Query Length by Relevance Class', fontweight='bold')
ax.set_xlabel('Class')
ax.set_ylabel('Word Count')
plt.suptitle('')
plt.tight_layout()
plt.savefig('outputs/figures/eda_query_len_by_class.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/figures/eda_query_len_by_class.png')

Saved: outputs/figures/eda_query_len_by_class.png


In [13]:
# ── Plot 4: Top query terms ────────────────────────────────────────────────────
stop = set(stopwords.words('english'))
all_words = ' '.join(df_raw['query'].dropna()).lower().split()
filtered = [w for w in all_words if w not in stop and len(w) > 2 and w.isalpha()]
top_words = Counter(filtered).most_common(20)
words, freqs = zip(*top_words)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(list(reversed(words)), list(reversed(freqs)), color='#4C72B0')
ax.set_title('Top 20 Query Terms (stopwords removed)', fontweight='bold')
ax.set_xlabel('Frequency')
plt.tight_layout()
plt.savefig('outputs/figures/eda_top_query_terms.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/figures/eda_top_query_terms.png')

Saved: outputs/figures/eda_top_query_terms.png


In [14]:
for label in ['E', 'S', 'C', 'I']:
    print(f"\n{'='*65}")
    print(f"  Class: {label_map[label]}")
    print('='*65)
    sample = df_raw[df_raw['esci_label'] == label][['query', 'product_title']].sample(2, random_state=42)
    for _, row in sample.iterrows():
        print(f"  Query : {row['query']}")
        print(f"  Title : {row['product_title']}")
        print()

gc.collect()


  Class: Exact
  Query : metal bed frame
  Title : Amazon Basics Foldable, 14" Black Metal Platform Bed Frame with Tool-Free Assembly, No Box Spring Needed - Twin

  Query : prime furniture
  Title : Signature Design by Ashley Chime 12" Medium Firm Memory Foam Mattress - CertiPUR-US Certified, Queen


  Class: Substitute
  Query : $10 candles
  Title : D'light Online 10" Elegant Taper Premium Quality Candles, Hand-Dipped, Dripless, Smokeless and Unwrapped Bulk Pack for Events - Set of 12 (10 Inch, Ivory)

  Query : xbox live gold
  Title : $10 Xbox Gift Card [Digital Code]


  Class: Complement
  Query : minibikes adult gas
  Title : Yoshimura TRC Full System Exhaust (Race/Stainless Steel/Carbon Fiber/Carbon Fiber)

  Query : 2ft christmas tree for kids
  Title : UMARDOO Christmas Tree Storage Bag - Xmas Tree Storage Container Stores Disassembled Artificial Christmas Tree,Durable Waterproof Zippered Bag with Carry Handles (Green, 65x15x30 in)


  Class: Irrelevant
  Query : *i'm not s

25

---
## 4. Preprocessing
> Cleans text, removes stopwords, builds TF-IDF features.

In [15]:
# Training configuration
BILSTM_EPOCHS     = 5
DISTILBERT_EPOCHS = 3
BATCH_SIZE        = 64


In [16]:
from src.preprocessing.preprocess import preprocess_texts, get_tfidf_features
from sklearn.model_selection import train_test_split

df = df_raw[['query', 'product_title', 'esci_label']].copy()
df = df.dropna(subset=['query', 'product_title', 'esci_label'])
df = df[df['query'].str.strip() != '']
df = df[df['product_title'].str.strip() != '']
df = df.reset_index(drop=True)

print(f'Clean rows: {len(df)}')
df = preprocess_texts(df)

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train: {len(train_df)} | Test: {len(test_df)}')
df[['query', 'query_clean', 'product_title', 'title_clean', 'esci_label', 'label']].head(3)

Clean rows: 20000
Train: 16000 | Test: 4000


,query,query_clean,product_title,title_clean,esci_label,label
0,bathroom fan without light,bathroom fan without light,Panasonic FV-20VQ3 WhisperCeiling 190 CFM Ceil...,panasonic fv 20vq3 whisperceiling 190 cfm ceil...,E,0
1,revent 80 cfm,revent 80 cfm,Homewerks 7141-80 Bathroom Fan Integrated LED ...,homewerks 7141 80 bathroom fan integrated led ...,E,0
2,revent 80 cfm,revent 80 cfm,Homewerks 7140-80 Bathroom Fan Ceiling Mount E...,homewerks 7140 80 bathroom fan ceiling mount e...,E,0


In [17]:
X_train_tfidf, X_test_tfidf, vectorizer = get_tfidf_features(
    train_df['text'], test_df['text'],
    max_features=20000
)
print(f'TF-IDF: train={X_train_tfidf.shape}, test={X_test_tfidf.shape}')
gc.collect()

TF-IDF: train=(16000, 20000), test=(4000, 20000)


0

---
## 5. Model Training & Evaluation

In [18]:
def show_figure(path):
    if os.path.exists(path):
        img = mpimg.imread(path)
        plt.figure(figsize=(12, 5))
        plt.imshow(img)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f'Figure not found: {path}')

all_metrics = []
per_class_f1 = {}
from sklearn.metrics import f1_score
from src.evaluation.evaluate import full_evaluation
print('Ready')

Ready


---
### 5.1 Model 1 — Naive Bayes

In [19]:
from src.models import naive_bayes

nb_model = naive_bayes.train(X_train_tfidf, train_df['label'].values)
nb_preds = naive_bayes.predict(nb_model, X_test_tfidf)
naive_bayes.save(nb_model, 'outputs/models/naive_bayes.joblib')

nb_metrics = full_evaluation(test_df['label'].values, nb_preds, 'Naive_Bayes')
all_metrics.append(nb_metrics)
f1s = f1_score(test_df['label'].values, nb_preds, average=None, zero_division=0)
per_class_f1['Naive_Bayes'] = f1s.tolist() if len(f1s) == 4 else np.pad(f1s, (0, 4-len(f1s))).tolist()


Classification Report — Naive_Bayes
              precision    recall  f1-score   support

       Exact       0.58      0.72      0.64      1000
  Substitute       0.57      0.44      0.49      1000
  Complement       0.86      0.89      0.87      1000
  Irrelevant       0.64      0.60      0.62      1000

    accuracy                           0.66      4000
   macro avg       0.66      0.66      0.66      4000
weighted avg       0.66      0.66      0.66      4000



In [20]:
show_figure('outputs/figures/confusion_matrix_Naive_Bayes.png')
show_figure('outputs/figures/f1_per_class_Naive_Bayes.png')

---
### 5.2 Model 2 — Logistic Regression

In [21]:
from src.models import logistic_regression

lr_model = logistic_regression.train(X_train_tfidf, train_df['label'].values)
lr_preds = logistic_regression.predict(lr_model, X_test_tfidf)
logistic_regression.save(lr_model, 'outputs/models/logistic_regression.joblib')

lr_metrics = full_evaluation(test_df['label'].values, lr_preds, 'Logistic_Regression')
all_metrics.append(lr_metrics)
f1s = f1_score(test_df['label'].values, lr_preds, average=None, zero_division=0)
per_class_f1['Logistic_Regression'] = f1s.tolist() if len(f1s) == 4 else np.pad(f1s, (0, 4-len(f1s))).tolist()


Classification Report — Logistic_Regression
              precision    recall  f1-score   support

       Exact       0.60      0.76      0.67      1000
  Substitute       0.60      0.48      0.54      1000
  Complement       0.88      0.89      0.89      1000
  Irrelevant       0.68      0.64      0.66      1000

    accuracy                           0.69      4000
   macro avg       0.69      0.69      0.69      4000
weighted avg       0.69      0.69      0.69      4000



In [22]:
show_figure('outputs/figures/confusion_matrix_Logistic_Regression.png')
show_figure('outputs/figures/f1_per_class_Logistic_Regression.png')

---
### 5.3 Model 3 — SVM (LinearSVC)

In [23]:
from src.models import svm

svm_model = svm.train(X_train_tfidf, train_df['label'].values)
svm_preds = svm.predict(svm_model, X_test_tfidf)
svm.save(svm_model, 'outputs/models/svm.joblib')

svm_metrics = full_evaluation(test_df['label'].values, svm_preds, 'SVM')
all_metrics.append(svm_metrics)
f1s = f1_score(test_df['label'].values, svm_preds, average=None, zero_division=0)
per_class_f1['SVM'] = f1s.tolist() if len(f1s) == 4 else np.pad(f1s, (0, 4-len(f1s))).tolist()


Classification Report — SVM
              precision    recall  f1-score   support

       Exact       0.63      0.76      0.69      1000
  Substitute       0.62      0.51      0.56      1000
  Complement       0.91      0.90      0.90      1000
  Irrelevant       0.68      0.68      0.68      1000

    accuracy                           0.71      4000
   macro avg       0.71      0.71      0.71      4000
weighted avg       0.71      0.71      0.71      4000



In [24]:
show_figure('outputs/figures/confusion_matrix_SVM.png')
show_figure('outputs/figures/f1_per_class_SVM.png')

---
### 5.4 Model 4 — BiLSTM
> Uses GloVe 300d pretrained embeddings. Download takes ~5 min. Skip if already downloaded.
> GPU recommended — ~10 min on T4.

In [25]:
import os
if not os.path.exists('glove/glove.6B.300d.txt'):
    print('Downloading GloVe embeddings (~860 MB)...')
    !wget -q --show-progress http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -q glove.6B.zip -d glove/
    !rm glove.6B.zip  # free disk space
    print('GloVe downloaded.')
else:
    print('GloVe already present, skipping download.')

glove.6B.zip        100%[===================>] 822.24M  4.97MB/s    in 2m 45s  
GloVe downloaded.


In [27]:
from src.models import bilstm

bilstm_model, bilstm_tokenizer = bilstm.train(
    train_df['text'].tolist(),
    train_df['label'].values,
    epochs=BILSTM_EPOCHS,
    embed_dim=300,
    batch_size=BATCH_SIZE,
    glove_path="glove/glove.6B.300d.txt",
)
bilstm_preds = bilstm.predict(bilstm_model, bilstm_tokenizer, test_df['text'].tolist())
bilstm.save(
    bilstm_model, bilstm_tokenizer,
    'outputs/models/bilstm.pt',
    'outputs/models/bilstm_tokenizer.joblib'
)

bilstm_metrics = full_evaluation(test_df['label'].values, bilstm_preds, 'BiLSTM')
all_metrics.append(bilstm_metrics)
f1s = f1_score(test_df['label'].values, bilstm_preds, average=None, zero_division=0)
per_class_f1['BiLSTM'] = f1s.tolist() if len(f1s) == 4 else np.pad(f1s, (0, 4-len(f1s))).tolist()

BiLSTM Epoch 5/5: 100%|██████████| 250/250 [00:03<00:00, 75.59it/s]



Classification Report — BiLSTM
              precision    recall  f1-score   support

       Exact       0.66      0.77      0.71      1000
  Substitute       0.61      0.60      0.61      1000
  Complement       0.92      0.89      0.91      1000
  Irrelevant       0.74      0.66      0.70      1000

    accuracy                           0.73      4000
   macro avg       0.73      0.73      0.73      4000
weighted avg       0.73      0.73      0.73      4000



In [28]:
show_figure('outputs/figures/confusion_matrix_BiLSTM.png')
show_figure('outputs/figures/f1_per_class_BiLSTM.png')

---
### 5.5 Model 5 — DistilBERT (fine-tuned)
> Input: `[CLS] query [SEP] product_title [SEP]`
> ~25 min on T4 GPU.

In [29]:
from src.models import distilbert

distilbert_model, distilbert_tokenizer = distilbert.train(
    train_df['query_clean'].tolist(),
    train_df['title_clean'].tolist(),
    train_df['label'].values,
    epochs=DISTILBERT_EPOCHS,
    batch_size=BATCH_SIZE,
)
distilbert_preds = distilbert.predict(
    distilbert_model, distilbert_tokenizer,
    test_df['query_clean'].tolist(),
    test_df['title_clean'].tolist(),
)
distilbert.save(distilbert_model, distilbert_tokenizer, 'outputs/models/distilbert')

distilbert_metrics = full_evaluation(test_df['label'].values, distilbert_preds, 'DistilBERT')
all_metrics.append(distilbert_metrics)
f1s = f1_score(test_df['label'].values, distilbert_preds, average=None, zero_division=0)
per_class_f1['DistilBERT'] = f1s.tolist() if len(f1s) == 4 else np.pad(f1s, (0, 4-len(f1s))).tolist()

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
DistilBERT Epoch 3/3: 100%|██████████| 250/250 [02:37<00:00,  1.59it/s]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Classification Report — DistilBERT
              precision    recall  f1-score   support

       Exact       0.65      0.77      0.70      1000
  Substitute       0.59      0.56      0.57      1000
  Complement       0.86      0.81      0.84      1000
  Irrelevant       0.73      0.69      0.71      1000

    accuracy                           0.71      4000
   macro avg       0.71      0.71      0.71      4000
weighted avg       0.71      0.71      0.71      4000



In [30]:
show_figure('outputs/figures/confusion_matrix_DistilBERT.png')
show_figure('outputs/figures/f1_per_class_DistilBERT.png')

---
## 6. Comparative Analysis

In [31]:
from src.evaluation.evaluate import plot_comparative_analysis, plot_f1_heatmap, save_comparative_summary

plot_comparative_analysis(all_metrics)
plot_f1_heatmap(per_class_f1)
save_comparative_summary(all_metrics)


COMPARATIVE SUMMARY
              model  accuracy  precision_weighted  recall_weighted  f1_weighted  f1_macro
        Naive_Bayes    0.6627              0.6606           0.6627       0.6573    0.6573
Logistic_Regression    0.6933              0.6941           0.6933       0.6895    0.6895
                SVM    0.7085              0.7095           0.7085       0.7061    0.7061
             BiLSTM    0.7338              0.7334           0.7338       0.7319    0.7319
             BiLSTM    0.7300              0.7335           0.7300       0.7302    0.7302
         DistilBERT    0.7053              0.7085           0.7053       0.7052    0.7052


In [32]:
show_figure('outputs/figures/comparative_analysis.png')
show_figure('outputs/figures/f1_heatmap_all_models.png')

In [38]:
# Summary table
summary_df = pd.DataFrame(all_metrics).sort_values('f1_weighted', ascending=False).reset_index(drop=True)
print('\n=== FINAL RESULTS (sorted by F1 weighted) ===')
display(summary_df.style.highlight_max(subset=summary_df.columns[1:], axis=0, color='#c6efce').format(
    {col: '{:.4f}' for col in summary_df.columns if col != 'model'}
))


=== FINAL RESULTS (sorted by F1 weighted) ===


,model,accuracy,precision_weighted,recall_weighted,f1_weighted,f1_macro
0,BiLSTM,0.7338,0.7334,0.7338,0.7319,0.7319
1,BiLSTM,0.7300,0.7335,0.7300,0.7302,0.7302
2,SVM,0.7085,0.7095,0.7085,0.7061,0.7061
3,DistilBERT,0.7053,0.7085,0.7053,0.7052,0.7052
4,Logistic_Regression,0.6933,0.6941,0.6933,0.6895,0.6895
5,Naive_Bayes,0.6627,0.6606,0.6627,0.6573,0.6573


---
## 7. Push Results to GitHub
> Add your GitHub PAT as a Colab secret:
> **Secrets (🔑 icon in left sidebar) → Add secret → Name: `GITHUB_PAT`**

In [34]:
!git add outputs/figures/ outputs/reports/
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   outputs/figures/comparative_analysis.png
	modified:   outputs/figures/confusion_matrix_BiLSTM.png
	modified:   outputs/figures/confusion_matrix_DistilBERT.png
	modified:   outputs/figures/eda_class_distribution.png
	modified:   outputs/figures/f1_heatmap_all_models.png
	modified:   outputs/figures/f1_per_class_BiLSTM.png
	modified:   outputs/figures/f1_per_class_DistilBERT.png
	modified:   outputs/reports/classification_report_BiLSTM.csv
	modified:   outputs/reports/classification_report_DistilBERT.csv
	modified:   outputs/reports/comparative_summary.csv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	glove/



In [35]:
!git commit -m "Add results: figures and classification reports"

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@ec3f46ddfdb1.(none)')


In [36]:
from google.colab import userdata
token = userdata.get('GITHUB_PAT')
repo = 'tp-0604/query-product-relevance-classification' # replace with your repository
!git remote set-url origin https://{token}@github.com/{repo}.git
!git push origin main

Everything up-to-date


---
## 8. Download Outputs Locally
> Alternative if you prefer not to push to GitHub.

In [37]:
import shutil
from google.colab import files

shutil.make_archive('/content/outputs', 'zip', 'outputs')
files.download('/content/outputs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>